# Proceso ETL con Polars y PostgreSQL

In [1]:
import polars as pl
from pathlib import Path
import warnings
import sqlalchemy
from sqlalchemy import create_engine
from sqlalchemy import text

## Leer archivos XLSX

In [2]:
# ruta_base = '/home/jovyan/data/spcia/'
ruta_base = './data/'
acciones_path = ruta_base + '/Acciones.xlsx'
auditorias_path = ruta_base + '/Auditorias.xlsx'
impactos_aud_path = ruta_base + '/impactos_aud.xlsx'
impactos_path = ruta_base + '/impactos.xlsx'

In [3]:
def extract_from_file(root_path: Path) -> pl.DataFrame:
    """Read XLSX files and construct DataFrames."""

    warnings.filterwarnings(
        "ignore", message="Workbook contains no default style"
    )

    try:
        df = pl.read_excel(source=root_path, engine="openpyxl")
        # display(df.head(2))
        return df

    except FileNotFoundError:
        print("No se encontró el archivo")

    except Exception as e:
        print(f"Error al leer el archivo: {e}")    

In [4]:
df_auditorias = extract_from_file(auditorias_path)

In [5]:
df_impactos_aud = extract_from_file(impactos_aud_path)

In [6]:
df_acciones = extract_from_file(acciones_path)

In [7]:
df_impactos = extract_from_file(impactos_path)

## DataFrames unidos

In [8]:
fact_auditoria = df_auditorias.join(
    df_impactos_aud,
    left_on="LLAVE_IMP_AUDIT",
    right_on="LLAVE_IMP_AUDIT",
    how="left"
)

print(f"Las dimesiones del DataFrame 'fact_auditoria' son: {fact_auditoria.shape}")

Las dimesiones del DataFrame 'fact_auditoria' son: (31264, 45)


In [9]:
fact_accion = df_acciones.join(
    df_impactos,
    left_on="LLAVE_IMP",
    right_on="ACC_LLAVE_IMP",
    how="left"
)

print(f"Las dimesiones del DataFrame 'fact_accion' son: {fact_accion.shape}")

Las dimesiones del DataFrame 'fact_accion' son: (192529, 30)


Se agregan columnas incrementales que serán usadas para las primary keys.

In [10]:
def add_incremental_col(df: pl.DataFrame, new_col: str) -> pl.DataFrame:
    """Add an incremental column"""
    return df.with_columns(
        pl.arange(1, df.height + 1).alias(new_col)
    )

In [11]:
fact_auditoria = add_incremental_col(fact_auditoria, 'auditoria_key')
fact_accion = add_incremental_col(fact_accion, 'accion_key')

In [12]:
print(f"{len(fact_auditoria.columns)} columnas de 'fact_auditoria':", fact_auditoria.columns)
print(f"{len(fact_accion.columns)} columnas de 'fact_accion':", fact_accion.columns)

46 columnas de 'fact_auditoria': ['LLAVE', 'LLAVE2', 'LLAVE_IMP', 'LLAVE_IMP_AUDIT', 'Año Cuenta Pública', 'Año Ejecución', 'AUD_NUMERO', 'Número de Auditoría', 'AUD_PERIODO_MULTIPLE', 'AUD_GPO_FUNCIONAL_CVE', 'Grupo Funcional', 'AUD_EFIS_SIGLAS', 'Ente Fiscalizado', 'Ente Fiscalizado PAAF', 'AUD_EFIS_CVE_DEPEND', 'AUD_EFIS_CVE_TIPO_ENTIDAD', 'AUD_EFIS_CVE_ENTIDAD', 'Tipo Entidad', 'AUD_SECTOR_CVE', 'AUD_SECTOR_CONSEC', 'Sector', 'Título', 'Título.', 'TITULOAPOST', 'Universo', 'Muestra', 'AUD_TIPO_AUD_SIGLAS', 'Tipo Auditoría', 'AUD_DICTAMEN_SIGLAS', 'Dictamen', 'AUD_EDO_LATITUD', 'AUD_EDO_LONGITUD', 'Entidad Federativa', 'AUD_LLAVE_ENTIDAD', 'Fondo1', 'AUD_LLAVE_FONDO', 'LIGA', 'Etapa', 'Área', 'AUD_ORIGEN', 'AUD_ORIGEN_SIGLAS', 'SI', 'Recuperado enAUD', 'Determinado Aud', 'enterada Aud', 'auditoria_key']
31 columnas de 'fact_accion': ['LLAVE_IMP', 'ACC_LLAVE', 'ACC_LLAVE_SV', 'ACC_LLAVE_IMP', 'Clave Acción', 'ACC_TIPO_SIGLAS', 'Texto Acción', 'Tipo Acción', 'Estado de Trámite', 'ACC_

## Dimensiones
A continuación se procede a crear las dimensiones (tablas catálogo) y extraer las columnas correspondientes de los Dataframes originales.

In [13]:
def dim_from_cols(df: pl.DataFrame,
                  extract_cols: list[str],
                  id_extracted_cols: str) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Create a catalog (dimention) by extracting columns from the received DataFrame.
    The generated catalog and the original DataFrame without the extracted columns are returned."""
    df_catalog = (
        df
        .select(extract_cols)
        .unique()
        .sort(extract_cols)
        .with_columns(
            pl.arange(1, pl.len() + 1).alias(id_extracted_cols)
        )
        .select([id_extracted_cols] + extract_cols)
    ) 

    df_fact = (
        df
        .join(df_catalog, on=extract_cols, how="left")
        .drop(extract_cols)
    )
    
    return df_fact, df_catalog

### DIM_AREA
**Nota:** La columna 'Nombre_Area' para crear DIM_AREA no está. Se usará 'Área'.

In [14]:
# Create dim_area
columns_to_be_extracted = ["Área", "AUD_ORIGEN"]

fact_auditoria, dim_area = dim_from_cols(
                    fact_auditoria,
                    columns_to_be_extracted,
                    "area_key")

### DIM_TIPO_AUD

In [15]:
# Create dim_tipo_aud
columns_to_be_extracted = ["Tipo Auditoría", "Grupo Funcional"]

fact_auditoria, dim_tipo_aud = dim_from_cols(
                    fact_auditoria,
                    columns_to_be_extracted,
                    "tipo_aud_key")

### DIM_ORIGEN_AUD
**Nota:** Las columnas 'Origen' y 'Descripcion_Origen' para crear DIM_ORIGEN_AUD no están.

### DIM_TIPO_ACC

In [16]:
print(f"{len(fact_accion.columns)} columnas de 'fact_accion':", fact_accion.columns)

31 columnas de 'fact_accion': ['LLAVE_IMP', 'ACC_LLAVE', 'ACC_LLAVE_SV', 'ACC_LLAVE_IMP', 'Clave Acción', 'ACC_TIPO_SIGLAS', 'Texto Acción', 'Tipo Acción', 'Estado de Trámite', 'ACC_STATUS_SIGLAS', 'CC DG Seguimiento', 'DG Seguimiento', 'Estado de Trámite SICSA', 'Estado Interno SICSA ', 'ACC_TIPO_STATUS_LLAVE', 'ACC_TIPO_STATUS', 'ACC_ORIGEN_LLAVE', 'Origen', 'Origen Acción', 'Orden_edo', 'Ente Observado', 'Tipo Entidad Acc', 'Recuperado en', 'FECH_LLAVE_ACCION', 'Actualizado', 'Determinado', 'enterada', 'Just', 'Sust', 'Por Recuperar', 'accion_key']


In [17]:
# Create dim_tipo_acc
columns_to_be_extracted = ["Tipo Acción", "ACC_TIPO_SIGLAS"]

fact_accion, dim_tipo_acc = dim_from_cols(
                    fact_accion,
                    columns_to_be_extracted,
                    "tipo_acc_key")

### DIM_ESTADO

In [18]:
# Create dim_estado
columns_to_be_extracted = ["Estado de Trámite", "Estado de Trámite SICSA", "ACC_STATUS_SIGLAS"]

fact_accion, dim_estado = dim_from_cols(
                    fact_accion,
                    columns_to_be_extracted,
                    "estado_key")

### DIM_ORIGEN_ACC

In [19]:
# Create dim_origen_acc
columns_to_be_extracted = ["Origen", "Origen Acción"]

fact_accion, dim_origen_acc = dim_from_cols(
                    fact_accion,
                    columns_to_be_extracted,
                    "origen_acc_key")

## DIM_TIEMPO y DIM_ENTIDAD
El Dataframe 'fact_accion' no tiene las columnas 'Año Cuenta Pública' ni 'Año Ejecución', por lo tanto, se usarán sólo los valores únicos de 'fact_auditoria'.

### DIM_TIEMPO

In [20]:
# Create dim_tiempo
columns_to_be_extracted = ["Año Cuenta Pública", "Año Ejecución"]

fact_auditoria, dim_tiempo = dim_from_cols(
                    fact_auditoria,
                    columns_to_be_extracted,
                    "tiempo_key")

Ahora se debe hacer un left join de ambos Dataframes para agregar la columna 'tipo_key' a 'FACT_ACCION'.

In [21]:
# Left join to add column 'tipo_key'
fact_accion = fact_accion.join(
    fact_auditoria.select(['LLAVE_IMP','tiempo_key']),
    left_on="LLAVE_IMP",
    right_on="LLAVE_IMP",
    how="left"
)

In [22]:
print(f"{len(fact_auditoria.columns)} columnas de 'fact_auditoria':", fact_auditoria.columns)
print(f"{len(fact_accion.columns)} columnas de 'fact_accion':", fact_accion.columns)

43 columnas de 'fact_auditoria': ['LLAVE', 'LLAVE2', 'LLAVE_IMP', 'LLAVE_IMP_AUDIT', 'AUD_NUMERO', 'Número de Auditoría', 'AUD_PERIODO_MULTIPLE', 'AUD_GPO_FUNCIONAL_CVE', 'AUD_EFIS_SIGLAS', 'Ente Fiscalizado', 'Ente Fiscalizado PAAF', 'AUD_EFIS_CVE_DEPEND', 'AUD_EFIS_CVE_TIPO_ENTIDAD', 'AUD_EFIS_CVE_ENTIDAD', 'Tipo Entidad', 'AUD_SECTOR_CVE', 'AUD_SECTOR_CONSEC', 'Sector', 'Título', 'Título.', 'TITULOAPOST', 'Universo', 'Muestra', 'AUD_TIPO_AUD_SIGLAS', 'AUD_DICTAMEN_SIGLAS', 'Dictamen', 'AUD_EDO_LATITUD', 'AUD_EDO_LONGITUD', 'Entidad Federativa', 'AUD_LLAVE_ENTIDAD', 'Fondo1', 'AUD_LLAVE_FONDO', 'LIGA', 'Etapa', 'AUD_ORIGEN_SIGLAS', 'SI', 'Recuperado enAUD', 'Determinado Aud', 'enterada Aud', 'auditoria_key', 'area_key', 'tipo_aud_key', 'tiempo_key']
28 columnas de 'fact_accion': ['LLAVE_IMP', 'ACC_LLAVE', 'ACC_LLAVE_SV', 'ACC_LLAVE_IMP', 'Clave Acción', 'Texto Acción', 'CC DG Seguimiento', 'DG Seguimiento', 'Estado Interno SICSA ', 'ACC_TIPO_STATUS_LLAVE', 'ACC_TIPO_STATUS', 'ACC_ORI

**Nota:** El Dataframe 'fact_auditoria' no tiene las columnas 'Nombre Entidad'.

El Dataframe 'fact_accion' no tiene las columnas 'Nombre Entidad' ni 'Entidad Federativa' ni 'Tipo Entidad', por lo tanto, se usarán sólo los valores únicos de 'fact_auditoria'.

### DIM_ENTIDAD

In [23]:
# Create dim_entidad
columns_to_be_extracted = ["Tipo Entidad", "Entidad Federativa"]

fact_auditoria, dim_entidad = dim_from_cols(
                    fact_auditoria,
                    columns_to_be_extracted,
                    "entidad_key")

Ahora se debe hacer un left join de ambos Dataframes para agregar la columna 'entidad_key' a 'FACT_ACCION'.

In [24]:
# Left join to add column 'entidad_key'
fact_accion = fact_accion.join(
    fact_auditoria.select(['LLAVE_IMP','entidad_key']),
    left_on="LLAVE_IMP",
    right_on="LLAVE_IMP",
    how="left"
)

In [25]:
print(f"{len(fact_auditoria.columns)} columnas de 'fact_auditoria':", fact_auditoria.columns)
print(f"{len(fact_accion.columns)} columnas de 'fact_accion':", fact_accion.columns)

42 columnas de 'fact_auditoria': ['LLAVE', 'LLAVE2', 'LLAVE_IMP', 'LLAVE_IMP_AUDIT', 'AUD_NUMERO', 'Número de Auditoría', 'AUD_PERIODO_MULTIPLE', 'AUD_GPO_FUNCIONAL_CVE', 'AUD_EFIS_SIGLAS', 'Ente Fiscalizado', 'Ente Fiscalizado PAAF', 'AUD_EFIS_CVE_DEPEND', 'AUD_EFIS_CVE_TIPO_ENTIDAD', 'AUD_EFIS_CVE_ENTIDAD', 'AUD_SECTOR_CVE', 'AUD_SECTOR_CONSEC', 'Sector', 'Título', 'Título.', 'TITULOAPOST', 'Universo', 'Muestra', 'AUD_TIPO_AUD_SIGLAS', 'AUD_DICTAMEN_SIGLAS', 'Dictamen', 'AUD_EDO_LATITUD', 'AUD_EDO_LONGITUD', 'AUD_LLAVE_ENTIDAD', 'Fondo1', 'AUD_LLAVE_FONDO', 'LIGA', 'Etapa', 'AUD_ORIGEN_SIGLAS', 'SI', 'Recuperado enAUD', 'Determinado Aud', 'enterada Aud', 'auditoria_key', 'area_key', 'tipo_aud_key', 'tiempo_key', 'entidad_key']
29 columnas de 'fact_accion': ['LLAVE_IMP', 'ACC_LLAVE', 'ACC_LLAVE_SV', 'ACC_LLAVE_IMP', 'Clave Acción', 'Texto Acción', 'CC DG Seguimiento', 'DG Seguimiento', 'Estado Interno SICSA ', 'ACC_TIPO_STATUS_LLAVE', 'ACC_TIPO_STATUS', 'ACC_ORIGEN_LLAVE', 'Orden_edo'

### Ejemplo de obtención de dimensiones (puede omitirse)
A continuación se muestra un ejemplo de cómo se extrajeron las columnas anteriores. Este ejemplo puede ser omitido.

In [26]:
df1 = pl.DataFrame({
    "anio_ejec": [2022, 2023, 2022, 2024],
    "mes_ejec": [1, 1, 2, 3],
    "monto": [100, 200, 150, 300]
})
print(df1)

shape: (4, 3)
┌───────────┬──────────┬───────┐
│ anio_ejec ┆ mes_ejec ┆ monto │
│ ---       ┆ ---      ┆ ---   │
│ i64       ┆ i64      ┆ i64   │
╞═══════════╪══════════╪═══════╡
│ 2022      ┆ 1        ┆ 100   │
│ 2023      ┆ 1        ┆ 200   │
│ 2022      ┆ 2        ┆ 150   │
│ 2024      ┆ 3        ┆ 300   │
└───────────┴──────────┴───────┘


In [27]:
columnas_catalogo = ["anio_ejec", "mes_ejec"]
df_resulting, dim = dim_from_cols(
                    df1,
                    columnas_catalogo,
                    "key")

In [28]:
df_recuperado = (
    df_resulting
    .join(dim, on="key", how="left")
    .select(df1.columns)
)
print(df1)
print(df_recuperado)

shape: (4, 3)
┌───────────┬──────────┬───────┐
│ anio_ejec ┆ mes_ejec ┆ monto │
│ ---       ┆ ---      ┆ ---   │
│ i64       ┆ i64      ┆ i64   │
╞═══════════╪══════════╪═══════╡
│ 2022      ┆ 1        ┆ 100   │
│ 2023      ┆ 1        ┆ 200   │
│ 2022      ┆ 2        ┆ 150   │
│ 2024      ┆ 3        ┆ 300   │
└───────────┴──────────┴───────┘
shape: (4, 3)
┌───────────┬──────────┬───────┐
│ anio_ejec ┆ mes_ejec ┆ monto │
│ ---       ┆ ---      ┆ ---   │
│ i64       ┆ i64      ┆ i64   │
╞═══════════╪══════════╪═══════╡
│ 2022      ┆ 1        ┆ 100   │
│ 2023      ┆ 1        ┆ 200   │
│ 2022      ┆ 2        ┆ 150   │
│ 2024      ┆ 3        ┆ 300   │
└───────────┴──────────┴───────┘


Los últmos Dataframes deben ser iguales.

## Conexión a PostgreSQL

In [29]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [30]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "10.0.18.2"
DB_PORT = 5439
DB_NAME = "asf_db"

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

In [31]:
# Primary keys of every table
primary_keys = {
    'fact_auditoria': 'auditoria_key',
    'fact_accion': 'accion_key',
    'dim_tiempo': 'tiempo_key',
    'dim_area': 'area_key',
    'dim_tipo_aud': 'tipo_aud_key',
    'dim_tipo_acc': 'tipo_acc_key',
    'dim_estado': 'estado_key',
    'dim_origen_acc': 'origen_acc_key',
    'dim_entidad': 'entidad_key'
}

# Foreign keys of every table ('table': [{'f_table': 'fk_column'}])
foreign_keys = {
    'fact_auditoria': [{'dim_tiempo': 'tiempo_key'},
                       {'dim_area': 'area_key'},
                       {'dim_tipo_aud': 'tipo_aud_key'},
                       {'dim_entidad': 'entidad_key'}
                        ],
    
    'fact_accion': [{'dim_tiempo': 'tiempo_key'},
                    {'dim_tipo_acc': 'tipo_acc_key'},
                    {'dim_estado': 'estado_key'},
                    {'dim_origen_acc': 'origen_acc_key'},
                    {'dim_entidad': 'entidad_key'}
                    ]
}

POSTGRES_SCHEMA = "bi_model_v1"

In [32]:
def qident(name: str) -> str:
    """Quote seguro para identificadores PostgreSQL."""
    return '"' + name.replace('"', '""') + '"'


def map_polars_to_postgres(colname: str, dtype: pl.DataType) -> str:
    """Mapea tipos Polars a PostgreSQL."""

    if dtype == pl.Date:
        return "DATE"

    if getattr(dtype, "base_type", lambda: None)() == pl.Datetime:
        return "TIMESTAMP"

    mapping = {
        pl.Int8: "SMALLINT",
        pl.Int16: "SMALLINT",
        pl.Int32: "INTEGER",
        pl.Int64: "BIGINT",
        pl.Boolean: "BOOLEAN",
        pl.Float32: "REAL",
        pl.Float64: "DOUBLE PRECISION",
        pl.Utf8: "TEXT",
        pl.String: "TEXT",
    }

    return mapping.get(dtype, "TEXT")

In [33]:
def create_table_from_df(
    engine,
    table_name: str,
    df: pl.DataFrame,
    primary_key: str | None = None,
    schema: str = POSTGRES_SCHEMA,
) -> None:
    """Crea una tabla PostgreSQL a partir del esquema de un Polars DataFrame."""
    schema_q = qident(schema)
    table_q = qident(table_name)
    full_name = f"{schema_q}.{table_q}"
    columns_sql = []

    for col, dtype in df.schema.items():
        sql_type = map_polars_to_postgres(col, dtype)
        columns_sql.append(f"{qident(col)} {sql_type}")

    pk_sql = ""
    if primary_key:
        pk_sql = f", CONSTRAINT {qident(f'pk_{table_name}')} PRIMARY KEY ({qident(primary_key)})"
    else:
        print(f"No se crea Primary Key para la tabla '{table_name}'")

    create_sql = f"""
    CREATE SCHEMA IF NOT EXISTS {schema_q};
    DROP TABLE IF EXISTS {schema_q}.{table_q};

    CREATE TABLE {full_name} (
        {", ".join(columns_sql)}
        {pk_sql}
    );
    """

    with engine.begin() as conn:
        conn.execute(text(create_sql))

In [34]:
def load_table_to_PostgreSQL(
    engine: sqlalchemy.engine.Engine,
    df: pl.DataFrame,
    table_name: str,
    primary_keys: dict[str, str] | None = None,
    batch_rows: int = 10_000,
    schema: str = POSTGRES_SCHEMA,
) -> None:
    """Load a Polars DataFrame to PostgreSQL using SQLAlchemy."""
    primary_keys = primary_keys or {}
    create_table_from_df(
        engine=engine,
        table_name=table_name,
        df=df,
        primary_key=primary_keys.get(table_name),
        schema=schema,
    )

    print(f"Subiendo tabla '{schema}.{table_name}' a PostgreSQL en lotes de {batch_rows:,} filas...")
    n = df.height
    step = 0
    try:
        for start in range(0, n, batch_rows):
            chunk = df.slice(start, batch_rows)
            chunk.to_pandas().to_sql(
                name=table_name,
                con=engine,
                schema=schema,
                if_exists="append",
                index=False,
                method="multi",
            )

            step += 1
            if step == 10:
                print(f"✅ Filas completadas: {min(start + batch_rows, n):,} de {n:,}")
                step = 0

        print(f"✅ Tabla '{schema}.{table_name}' subida correctamente.\n")

    except Exception as e:
        print(f"❌ Error cargando '{schema}.{table_name}': {e}")
        raise

In [35]:
TABLES_TO_PROCESS = {
    "fact_auditoria": fact_auditoria,
    "fact_accion": fact_accion,
    "dim_tiempo": dim_tiempo,
    "dim_area": dim_area,
    "dim_tipo_aud": dim_tipo_aud,
    "dim_tipo_acc": dim_tipo_acc,
    "dim_estado": dim_estado,
    "dim_origen_acc": dim_origen_acc,
    "dim_entidad": dim_entidad
}

In [36]:
for table_name, df in TABLES_TO_PROCESS.items():
    print("\n" + "=" * 25)
    print(f"📊 Procesando tabla: {table_name}")
    print("=" * 25)
    try:
        load_table_to_PostgreSQL(
            engine, df, table_name, primary_keys,
            10_000, POSTGRES_SCHEMA
        )

    except Exception as e:
        print(
            f"\n❌ FALLO CRÍTICO para {table_name}.\n")
        print(f"'{e}'")
        print("=" * 25)

print("\n--- Tablas subidas a PostgreSQL correctamente ---")


📊 Procesando tabla: fact_auditoria
Subiendo tabla 'bi_model_v1.fact_auditoria' a PostgreSQL en lotes de 10,000 filas...
✅ Tabla 'bi_model_v1.fact_auditoria' subida correctamente.


--- Tablas subidas a PostgreSQL correctamente ---

📊 Procesando tabla: fact_accion
Subiendo tabla 'bi_model_v1.fact_accion' a PostgreSQL en lotes de 10,000 filas...
✅ Filas completadas: 100,000 de 192,529
✅ Filas completadas: 192,529 de 192,529
✅ Tabla 'bi_model_v1.fact_accion' subida correctamente.


--- Tablas subidas a PostgreSQL correctamente ---

📊 Procesando tabla: dim_tiempo
Subiendo tabla 'bi_model_v1.dim_tiempo' a PostgreSQL en lotes de 10,000 filas...
✅ Tabla 'bi_model_v1.dim_tiempo' subida correctamente.


--- Tablas subidas a PostgreSQL correctamente ---

📊 Procesando tabla: dim_area
Subiendo tabla 'bi_model_v1.dim_area' a PostgreSQL en lotes de 10,000 filas...
✅ Tabla 'bi_model_v1.dim_area' subida correctamente.


--- Tablas subidas a PostgreSQL correctamente ---

📊 Procesando tabla: dim_tipo_a

In [37]:
def create_foreign_keys(
    engine,
    foreign_keys: dict,
    schema: str = POSTGRES_SCHEMA,
) -> None:
    """Create foreign keys in PostgreSQL."""
    def qident(name: str) -> str:
        return '"' + name.replace('"', '""') + '"'

    schema_q = qident(schema)
    with engine.begin() as conn:
        for child_table, fk_list in foreign_keys.items():
            child_q = qident(child_table)
            for fk_def in fk_list:
                for parent_table, fk_column in fk_def.items():
                    parent_q = qident(parent_table)
                    fk_col_q = qident(fk_column)
                    constraint_name = (
                        f"fk_{child_table}_{parent_table}"
                    )
                    constraint_q = qident(constraint_name)

                    alter_sql = f"""
                    ALTER TABLE {schema_q}.{child_q}
                    ADD CONSTRAINT {constraint_q}
                    FOREIGN KEY ({fk_col_q})
                    REFERENCES {schema_q}.{parent_q} ({fk_col_q});
                    """

                    try:
                        conn.execute(text(alter_sql))
                        print(
                            f"✅ FK creada: "
                            f"{child_table}.{fk_column} "
                            f"-> {parent_table}.{fk_column}"
                        )

                    except Exception as e:
                        print(
                            f"❌ Error creando FK "
                            f"{constraint_name}: {e}"
                        )
                        raise

In [38]:
create_foreign_keys(
    engine=engine, foreign_keys=foreign_keys, schema=POSTGRES_SCHEMA,
)

✅ FK creada: fact_auditoria.tiempo_key -> dim_tiempo.tiempo_key
✅ FK creada: fact_auditoria.area_key -> dim_area.area_key
✅ FK creada: fact_auditoria.tipo_aud_key -> dim_tipo_aud.tipo_aud_key
✅ FK creada: fact_auditoria.entidad_key -> dim_entidad.entidad_key
✅ FK creada: fact_accion.tiempo_key -> dim_tiempo.tiempo_key
✅ FK creada: fact_accion.tipo_acc_key -> dim_tipo_acc.tipo_acc_key
✅ FK creada: fact_accion.estado_key -> dim_estado.estado_key
✅ FK creada: fact_accion.origen_acc_key -> dim_origen_acc.origen_acc_key
✅ FK creada: fact_accion.entidad_key -> dim_entidad.entidad_key
